# HBR YouTube OCR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/Data_crawling/blob/main/hbr_youtube_ocr/hbr_youtube_ocr_colab.ipynb?skip_cache=true)

YouTube 영상 다운로드 → 전체 WAV 추출 → OpenCV 고정 ROI → MangaOCR → 화자/대사 timestamp 매칭.

Colab 링크는 `skip_cache=true`를 사용해 GitHub의 최신 노트북을 다시 불러옵니다.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U pytubefix manga-ocr opencv-python-headless pandas pillow


In [ ]:
YOUTUBE_URL = "https://www.youtube.com/watch?v=zHN25OVshIY" # @param {type:"string"}
OUTPUT_DIR = "/content/hbr_youtube_ocr" # @param {type:"string"}

from pathlib import Path
from pytubefix import YouTube
import subprocess

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

yt = YouTube(YOUTUBE_URL)
stream = yt.streams.filter(progressive=True, file_extension="mp4").order_by("resolution").desc().first()
if stream is None:
    raise RuntimeError("progressive MP4 stream not found")

downloaded = Path(stream.download(output_path=str(output_dir), filename="source.mp4"))
video_path = downloaded
audio_path = output_dir / "full_audio.wav"

subprocess.run([
    "ffmpeg", "-y", "-i", str(video_path),
    "-vn", "-c:a", "pcm_s16le", str(audio_path)
], check=True)

print("video:", video_path)
print("audio:", audio_path)


In [ ]:
PREVIEW_TIME_SEC = 60.0 # @param {type:"number"}
SPEAKER_X1 = 0.050 # @param {type:"number"}
SPEAKER_Y1 = 0.600 # @param {type:"number"}
SPEAKER_X2 = 0.190 # @param {type:"number"}
SPEAKER_Y2 = 0.695 # @param {type:"number"}
DIALOGUE_X1 = 0.155 # @param {type:"number"}
DIALOGUE_Y1 = 0.710 # @param {type:"number"}
DIALOGUE_X2 = 0.900 # @param {type:"number"}
DIALOGUE_Y2 = 0.820 # @param {type:"number"}

import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps
cap.set(cv2.CAP_PROP_POS_MSEC, PREVIEW_TIME_SEC * 1000)
ok, frame = cap.read()
cap.release()
if not ok:
    raise RuntimeError("preview frame read failed")

def to_box(x1, y1, x2, y2):
    return int(x1*width), int(y1*height), int(x2*width), int(y2*height)

speaker_box = to_box(SPEAKER_X1,SPEAKER_Y1,SPEAKER_X2,SPEAKER_Y2)
dialogue_box = to_box(DIALOGUE_X1,DIALOGUE_Y1,DIALOGUE_X2,DIALOGUE_Y2)

preview = frame.copy()
cv2.rectangle(preview, speaker_box[:2], speaker_box[2:], (0,255,0), 3)
cv2.rectangle(preview, dialogue_box[:2], dialogue_box[2:], (255,0,0), 3)
print(f"{width}x{height}, fps={fps:.3f}, duration={duration:.1f}s")
plt.figure(figsize=(16,9))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


In [ ]:
from manga_ocr import MangaOcr
from PIL import Image
import re

mocr = MangaOcr()

def crop_roi(img, box):
    x1,y1,x2,y2 = box
    return img[y1:y2, x1:x2]

def prep(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 170, 255, cv2.THRESH_BINARY)
    return Image.fromarray(cv2.bitwise_not(binary)).convert("RGB")

def norm(text):
    return re.sub(r"\s+", "", text).strip()

print("speaker:", norm(mocr(prep(crop_roi(frame, speaker_box)))))
print("dialogue:", norm(mocr(prep(crop_roi(frame, dialogue_box)))))


In [ ]:
SAMPLE_INTERVAL_SEC = 0.5 # @param {type:"number"}

from difflib import SequenceMatcher
import pandas as pd

def similar(a, b):
    if not a or not b:
        return False
    return a.startswith(b) or b.startswith(a) or SequenceMatcher(None,a,b).ratio() >= 0.82

cap = cv2.VideoCapture(str(video_path))
duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
rows = []
t = 0.0
while t <= duration:
    cap.set(cv2.CAP_PROP_POS_MSEC, t*1000)
    ok, img = cap.read()
    if not ok:
        break
    speaker = norm(mocr(prep(crop_roi(img, speaker_box))))
    dialogue = norm(mocr(prep(crop_roi(img, dialogue_box))))
    rows.append({"time_sec":round(t,3),"speaker":speaker,"dialogue":dialogue})
    t += SAMPLE_INTERVAL_SEC
cap.release()

raw_df = pd.DataFrame(rows)
raw_df.to_csv(output_dir/"ocr_raw_samples.csv", index=False, encoding="utf-8-sig")

segments = []
current = None
for row in rows:
    s, d, ts = row["speaker"], row["dialogue"], row["time_sec"]
    if not d:
        if current is not None:
            current["end_sec"] = ts
            segments.append(current)
            current = None
        continue
    if current is None:
        current = {"start_sec":ts,"end_sec":ts,"speaker":s,"dialogue":d}
        continue
    if (s == current["speaker"] or not s) and similar(d, current["dialogue"]):
        current["end_sec"] = ts
        if len(d) > len(current["dialogue"]):
            current["dialogue"] = d
        if not current["speaker"] and s:
            current["speaker"] = s
    else:
        segments.append(current)
        current = {"start_sec":ts,"end_sec":ts,"speaker":s,"dialogue":d}
if current is not None:
    segments.append(current)

segments_df = pd.DataFrame(segments)
segments_df.to_csv(output_dir/"dialogue_segments.csv", index=False, encoding="utf-8-sig")
display(segments_df.head(50))
